# The 2D Ising model

Each site of an $N \times N$ square lattice holds a spin $s_i = \pm 1$.
The lattice has periodic boundary conditions, so the right edge connects to the left edge and the top to the bottom.
The energy of a spin configuration is

$$H = -J \sum_{\langle ij \rangle} s_i s_j ,$$

where the sum runs over all pairs of nearest neighbors (each pair counted once).
With $J > 0$, parallel neighbors lower the energy.

At temperature $kT$, each configuration occurs with the Boltzmann probability $P \propto e^{-H/kT}$.
The model has a phase transition at the critical temperature $kT_c = 2J / \ln(1 + \sqrt{2}) \approx 2.269\,J$.
Below $kT_c$ the spins tend to align (the lattice is magnetized); above it they are disordered.

We track two quantities:
- the magnetization per spin, $M = \frac{1}{N^2} \sum_i s_i$, and
- the energy per spin, $E = H / N^2$ (in units of $J$; $-2$ when all spins are aligned).

## Sampling algorithms
Each algorithm is a Markov chain: a sequence of random updates whose configurations are, after enough updates, distributed according to the Boltzmann probability.
- **Heat bath**: pick a random spin and set it to up with probability $P_\uparrow = e^{-E_\uparrow/kT} / (e^{-E_\uparrow/kT} + e^{-E_\downarrow/kT})$, where $E_\uparrow$ and $E_\downarrow$ are the energies with that spin up or down (with its neighbors fixed).
- **Metropolis**: pick a random spin and flip it with probability $\min(1, e^{-\Delta E/kT})$, where $\Delta E$ is the change in energy from the flip.
- **Wolff**: grow a cluster of parallel spins from a random seed, adding each parallel neighbor with probability $p = 1 - e^{-2J/kT}$ per bond, then flip the whole cluster (see `WolffDemo.ipynb`).

For heat bath and Metropolis, $N^2$ single-spin updates make one **sweep**, in which each spin gets (on average) one chance to update.


### Setup
Install the required Python packages if this environment has not been set up yet.


In [ ]:
%pip install -r ../requirements.txt

### Imports
Load the plotting tools and the Ising helper functions (defined in `isingsim_funcs.py`).


In [ ]:
# Import the required libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import mplcursors

# Import the Ising helper functions
from isingsim_funcs import SampleGrid, CorrelationFun, RadialAverage, ClusterSizeStats, AnimateClusters

# Makes all interactive plots possible, both on VSCode and Jupyter notebooks:
%matplotlib widget

# %matplotlib qt
# Restart the kernel and uncomment this line to show plots in a separate window instead

### Parameters
Choose the lattice size, temperature, and sampling algorithm.
The temperature is set relative to the critical temperature $kT_c$: try values below, near, and above it.


In [ ]:
N = 100  # linear lattice size (N x N spins)
J = 1  # coupling strength
kT_c = 2 / np.log(1 + np.sqrt(2))  # critical temperature, about 2.269 (for J = 1)
kT = 0.85 * kT_c  # temperature (here below the critical temperature)

fractionUp = 0.5  # average fraction of +1 spins in the initial configuration
samplingMethod = 'HeatBath'  # 'HeatBath', 'Metropolis', or 'Wolff'

# Number of steps to run (numTimePoints), and how often to store the grid (everyT):
if samplingMethod in ['Metropolis', 'HeatBath']:
    # Each step updates one spin, so N^2 steps make one sweep of the lattice
    numTimePoints = 100 * N**2  # 100 sweeps
    everyT = N**2  # store once per sweep
    stepsPerTimeUnit, timeUnit = N**2, 'sweeps'
elif samplingMethod == 'Wolff':
    # Each step flips one cluster
    # (starting from a random grid, clusters are small at first and grow as the grid orders)
    numTimePoints = 20 * N
    everyT = N // 5
    stepsPerTimeUnit, timeUnit = 1, 'cluster flips'

### Run the simulation
Start from a random spin configuration and run the sampler.
`SampleGrid` returns the final grid, plus the energy, magnetization, and grid stored every `everyT` steps.


In [ ]:
# Random initial configuration: each spin is +1 with probability fractionUp
grid = np.sign(fractionUp - np.random.random((N, N)))

# Run the sampler
finalGrid, energies, magnetizations, gridHistory = SampleGrid(
    grid, kT, J, numTimePoints, everyT, samplingMethod)

# Time of each stored sample:
times = np.arange(len(energies)) * everyT / stepsPerTimeUnit

### Animation
Watch the stored spin configurations evolve (black = up, white = down).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(gridHistory[0], cmap='binary', vmin=-1, vmax=1)
ax.set_title(f'2D Ising Model with {N} by {N} lattice', size=15)
plt.axis('off')

# Text annotation for energy and magnetization
time_text = ax.text(0.15, -0.04, '', transform=ax.transAxes, color='k', fontsize=15)

def update(i):
    im.set_array(gridHistory[i])
    time_text.set_text(f'J = {J}, kT = {kT:.2f}, M = {magnetizations[i]:.3f}, E = {energies[i]:.3f}')
    return im, time_text

ani = FuncAnimation(fig, update, frames=len(gridHistory),
                    interval=200, blit=True, repeat=False)  # Change speed by varying "interval"

# To also save the animation as a gif, uncomment this:
# writer = PillowWriter(fps=15)
# ani.save('ising.gif', writer=writer)

### Energy and magnetization
Plot the energy and magnetization over time, with histograms of their values.
The chain has reached equilibrium once these stop drifting and fluctuate about steady values; only samples after that point represent the Boltzmann distribution.
Below $kT_c$, the magnetization settles near $+M$ or $-M$ (and a Wolff cluster flip can switch between them).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
plt.subplots_adjust(wspace=0.3)

ax_en, ax_en_hist = axes[0]
ax_mag, ax_mag_hist = axes[1]

custom_blue = (0, 0.4470, 0.7410)

# Plot energy
ax_en.plot(times, energies, color=custom_blue)
ax_en.set_xlabel(f'time ({timeUnit})')
ax_en.set_ylabel('energy per spin')

# Plot energy histogram
ax_en_hist.hist(energies, bins='sturges', color=custom_blue, alpha=0.7, edgecolor='black')
ax_en_hist.set_xlabel('energy per spin')
ax_en_hist.set_ylabel('frequency')

# Plot magnetization
ax_mag.plot(times, magnetizations, color=custom_blue)
ax_mag.set_xlabel(f'time ({timeUnit})')
ax_mag.set_ylabel('magnetization per spin')

# Plot magnetization histogram
ax_mag_hist.hist(magnetizations, bins='sturges', color=custom_blue, alpha=0.7, edgecolor='black')
ax_mag_hist.set_xlabel('magnetization per spin')
ax_mag_hist.set_ylabel('frequency')

plt.show()

# Enable cursors upon clicking on a point:
cursor = mplcursors.cursor(ax_en.get_lines(), hover=False)
cursor = mplcursors.cursor(ax_mag.get_lines(), hover=False)

### Correlations
The connected correlation function of the final grid measures how strongly two spins a displacement $(\Delta x, \Delta y)$ apart point the same way, beyond what the overall magnetization explains:

$$C(\Delta x, \Delta y) = \langle s(x, y)\, s(x + \Delta x, y + \Delta y) \rangle - \langle s \rangle^2 ,$$

averaged over all sites $(x, y)$.
Averaging over directions gives $C(r)$ as a function of distance $r$ alone.
How quickly $C(r)$ decays sets the correlation length: the typical size of aligned regions.


In [ ]:
fig = plt.figure(figsize=(10, 7))
plt.subplots_adjust(wspace=0.4)

ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122)

# Connected correlation function, with zero displacement at the center:
corrMatrix = CorrelationFun(finalGrid)

# Displacements (dx, dy) matching the entries of corrMatrix:
d = np.arange(N) - N // 2
X, Y = np.meshgrid(d, d)

ax1.plot_surface(X, Y, corrMatrix, cmap='coolwarm')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('Corr')
ax1.set_title('Correlation', fontweight='bold')

# Average over directions, giving the correlation as a function of distance:
R = RadialAverage(corrMatrix, N)
ax2.plot(R)
ax2.set_xlabel('Distance')
ax2.set_ylabel('Correlation')
ax2.set_title('Radial average', fontweight='bold')

plt.show()

# Add cursor to click over line plot and get coordinate info
cursor = mplcursors.cursor(ax2.get_lines(), hover=False)

### Cluster sizes
Find the clusters (connected regions of parallel spins) in the final grid, and plot the distribution of their sizes on log-log axes.
Near $kT_c$, clusters of all sizes appear, and the distribution approaches a power law.


In [ ]:
clusterSizes = ClusterSizeStats(finalGrid)

### Clusters at the critical point
At the critical temperature $kT_c$ there is no typical cluster size: clusters of every size appear, from single spins up to clusters spanning the lattice.
The cluster-size distribution then follows a power law, $P(s) \propto s^{-\alpha}$, which is a straight line on log-log axes.

The cell below starts from a random grid and runs heat-bath sampling at exactly $kT_c$, storing the grid every 5 sweeps (this takes about 30–60 s).
The animation shows the clusters merging and growing as the system approaches the critical state.
Each cluster is colored by its direction and size, so the largest clusters light up.
The distributions from earlier frames stay on the plot in light gray, so you can watch the distribution straighten into a power law.

Compare with temperatures below and above $kT_c$ (e.g., `0.9 * kT_c` and `1.2 * kT_c`): is there still a power law?


In [ ]:
Nc = 150  # linear lattice size for this run
critGrid = np.sign(0.5 - np.random.random((Nc, Nc)))  # random initial configuration

# Run heat-bath sampling at the critical temperature, storing the grid every 5 sweeps:
_, _, _, critHistory = SampleGrid(critGrid, kT_c, J, 300 * Nc**2, 5 * Nc**2, 'HeatBath')

# Animate the clusters and their size distribution:
aniClusters = AnimateClusters(critHistory)